# Demo 2 — Technique Comparison
## Session 5: Core Prompt Engineering Techniques

**Goal:** Run the same bug triage task through 5 prompting techniques, score each with an LLM-as-judge, and compare results against cost.

**Task:** Classify a bug report -> severity (P0-P3) + owning team + first investigation step.

**Ground truth:** P0, Notifications team (email pipeline not processing Stripe webhooks).

**What to watch:** The prompt displayed before each technique shows exactly what the model receives. Notice how structure and examples change what the model produces.

**Expected outcome:**
- Zero-Shot: baseline — fast, cheap, but may miss severity/team
- CoT: adds visible reasoning steps
- Few-Shot: model learns format and label space from examples
- Few-Shot + CoT: best of both — format AND reasoning
- Self-Consistency (N=5): most reliable but 5x cost

The **Complexity Ladder** rule: start at Zero-Shot, escalate only with evidence.

In [1]:
# -- Setup -----------------------------------------------------------------
import os, json, re, textwrap
import pandas as pd
from collections import Counter
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display

load_dotenv()

client = OpenAI(base_url=os.environ["API_BASE_URL"], api_key=os.environ["API_KEY"])
MODEL = os.environ.get("MODEL_NAME", "claude-sonnet-4-6")

# -- Shared task (identical across all 5 techniques) -----------------------
BUG_REPORT = """Title: Payment confirmation email not sent after checkout
Description: Users complete purchase but receive no confirmation email.
Steps: Add to cart -> checkout -> pay via Stripe -> order 'confirmed' in dashboard
  but no email arrives. Stripe webhooks show 'charge.succeeded'.
Frequency: ~40% of transactions in the last 24 hours.
Impact: Customer support receiving 80+ tickets/hour."""

GROUND_TRUTH = {"severity": "P0", "owner_team": "Notifications",
                "root_cause": "Email pipeline not processing Stripe webhook events"}

SYSTEM = """You are a senior software engineer triaging production bugs.

Severity: P0=revenue impact >20% users, P1=major feature broken,
P2=degraded <5% users or workaround exists, P3=cosmetic.

Teams: Payments (Stripe/billing), Notifications (email/SMS/SendGrid),
Platform (infra/queues), Customer Experience (support)."""

outputs = {}

def show_prompt(system, user, technique):
    w = 68
    print(f" TECHNIQUE: {technique} ".center(w, "="))
    print("[ SYSTEM ]")
    for line in system.strip().split("\n"):
        print(f"  {line}")
    print()
    print("[ USER ]")
    for line in user.strip().split("\n"):
        print(f"  {line}")
    print("=" * w)
    print()

def chat(system, user, max_tokens=600, temperature=1.0, json_mode=False):
    kwargs = dict(model=MODEL, max_tokens=max_tokens, temperature=temperature,
                  messages=[{"role": "system", "content": system},
                             {"role": "user",   "content": user}])
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    return client.chat.completions.create(**kwargs).choices[0].message.content

print(f"Model: {MODEL}")
print(f"Ground truth: severity={GROUND_TRUTH['severity']}, team={GROUND_TRUTH['owner_team']}")

Model: llama3.2:1b
Ground truth: severity=P0, team=Notifications


## Technique 1 — Zero-Shot
**What it is:** Instruction + input only. No examples. Model draws on training knowledge.
**Cost:** 1x (baseline)
**When to use:** Always start here.

In [2]:
# TECHNIQUE 1: Zero-Shot  |  COST: 1x baseline
system = SYSTEM
user = f"""Triage this bug. Give: severity (P0-P3), owner team, root cause hypothesis, first step.

{BUG_REPORT}"""

show_prompt(system, user, "1 - Zero-Shot")
outputs["Zero-Shot"] = chat(system, user, max_tokens=400)
print("-- RESPONSE --")
print(outputs["Zero-Shot"])

===================== TECHNIQUE: 1 - Zero-Shot =====================
[ SYSTEM ]
  You are a senior software engineer triaging production bugs.
  
  Severity: P0=revenue impact >20% users, P1=major feature broken,
  P2=degraded <5% users or workaround exists, P3=cosmetic.
  
  Teams: Payments (Stripe/billing), Notifications (email/SMS/SendGrid),
  Platform (infra/queues), Customer Experience (support).

[ USER ]
  Triage this bug. Give: severity (P0-P3), owner team, root cause hypothesis, first step.
  
  Title: Payment confirmation email not sent after checkout
  Description: Users complete purchase but receive no confirmation email.
  Steps: Add to cart -> checkout -> pay via Stripe -> order 'confirmed' in dashboard
    but no email arrives. Stripe webhooks show 'charge.succeeded'.
  Frequency: ~40% of transactions in the last 24 hours.
  Impact: Customer support receiving 80+ tickets/hour.



-- RESPONSE --
**Triage Outcome:** P2 - Degraded (<5%)

**Owner Team:** Payments (Stripe/billing)

**Root Cause Hypothesis:** The payment confirmation email is not being sent because Stripe's webhooks are not triggering, despite the transaction being confirmed.

**First Step:**

1. Log in to the Stripe dashboard of the Payment gateway.
2. Navigate to the "Webhooks" tab and find the webhook related to the specific product (e.g., `charge.succeeded`).
3. Check if any relevant webhooks are enabled for this endpoint.
4. Make sure there's no additional configuration or error that might prevent it from triggering.

**Additional Steps:**

1. Verify that the customer has a valid payment method and email address associated with their Stripe account.
2. Log in to the Support platform, navigate to the "Tickets" tab, and create a new ticket for this bug report. Use the following keywords:
	* Status: New
	* Report type: Bug
	* Description: Payment confirmation email not sent after checkout
3. Reques

## Technique 2 — Zero-Shot CoT
**What it is:** Append "Let's think step by step" — triggers intermediate reasoning tokens.
**Why it works:** Each reasoning token conditions the next token (autoregressive mechanism). Intermediate steps = computation graph.
**Benchmark:** MultiArith 17.7% -> 78.7% (Kojima et al. 2022)
**Cost:** ~1.5x (more output tokens)

In [3]:
# TECHNIQUE 2: Zero-Shot CoT  |  COST: ~1.5x
system = SYSTEM
user = f"""Triage this bug. Give: severity (P0-P3), owner team, root cause hypothesis, first step.

{BUG_REPORT}

Let's think step by step:"""

show_prompt(system, user, "2 - Zero-Shot CoT")
outputs["Zero-Shot CoT"] = chat(system, user, max_tokens=600)
print("-- RESPONSE --")
print(outputs["Zero-Shot CoT"])

=================== TECHNIQUE: 2 - Zero-Shot CoT ===================
[ SYSTEM ]
  You are a senior software engineer triaging production bugs.
  
  Severity: P0=revenue impact >20% users, P1=major feature broken,
  P2=degraded <5% users or workaround exists, P3=cosmetic.
  
  Teams: Payments (Stripe/billing), Notifications (email/SMS/SendGrid),
  Platform (infra/queues), Customer Experience (support).

[ USER ]
  Triage this bug. Give: severity (P0-P3), owner team, root cause hypothesis, first step.
  
  Title: Payment confirmation email not sent after checkout
  Description: Users complete purchase but receive no confirmation email.
  Steps: Add to cart -> checkout -> pay via Stripe -> order 'confirmed' in dashboard
    but no email arrives. Stripe webhooks show 'charge.succeeded'.
  Frequency: ~40% of transactions in the last 24 hours.
  Impact: Customer support receiving 80+ tickets/hour.
  
  Let's think step by step:



-- RESPONSE --
**Severity:** P1 (major feature broken)

**Owner Team:** Payments

**Root Cause Hypothesis:**

The root cause hypothesis is that the payment confirmation email not being sent after checkout may be due to an incorrect order status or a webhook setup issue. This is likely because Stripe's webhooks are triggered when a charge is successful, but the order dashboard shows a "unconfirmed" status.

**First Step:**

To investigate this further, let's create a new endpoint in our payments API that triggers a Stripe webhook for each payment method (e.g., credit card, PayPal). We can then verify if any webhook events are being triggered when a charge is successful.

**Next Steps:**

1. Create an API endpoint in Stripe to trigger the webhook for order creation:
```python
import stripe

stripe.api_key = 'your_stripe_api_key'

def create_webhook_endpoint(base_path, method):
    # Define the Stripe API version and client secret
    clientsecret = 'your_stripe_client_secret'

    signat

## Technique 3 — Few-Shot
**What it is:** Include 3 input->output demonstration pairs. The model infers the pattern.
**Why it works:** In-Context Learning (ICL) — model learns format AND label space from examples.
**Min et al. 2022 surprise:** Even random labels cause only 0-5% accuracy drop. What matters is FORMAT and LABEL SPACE, not label correctness.
**Cost:** ~2.5x (example tokens added to input)

> **Note:** The examples below are *demonstrations* of the output format — they are NOT the bug to triage. The target bug comes after the `---` separator.

In [4]:
# TECHNIQUE 3: Few-Shot  |  COST: ~2.5x
# Fix: clear separator between examples and target to prevent model from triaging all bugs
system = SYSTEM
user = f"""Below are examples showing how to triage a bug report.
Study the format, then triage only the bug marked TARGET.

--- EXAMPLES (do not triage these) ---

Bug: Login page returns 500 for all users. DB connection pool exhausted.
Triage: P0 | Platform | DB pool not scaling under load | Check RDS max_connections

Bug: Product images not loading on iOS 17 Safari. CSS transform issue.
Triage: P2 | Frontend | WebKit rendering regression in iOS 17 | Disable CSS transforms and retest

Bug: Password reset emails arrive 45 min late. SendGrid shows queued.
Triage: P1 | Notifications | SendGrid outbound queue congested | Check SendGrid status + queue depth

--- TARGET (triage this one) ---

Bug: {BUG_REPORT}
Triage:"""

show_prompt(system, user, "3 - Few-Shot")
outputs["Few-Shot"] = chat(system, user, max_tokens=400)
print("-- RESPONSE --")
print(outputs["Few-Shot"])

===================== TECHNIQUE: 3 - Few-Shot ======================
[ SYSTEM ]
  You are a senior software engineer triaging production bugs.
  
  Severity: P0=revenue impact >20% users, P1=major feature broken,
  P2=degraded <5% users or workaround exists, P3=cosmetic.
  
  Teams: Payments (Stripe/billing), Notifications (email/SMS/SendGrid),
  Platform (infra/queues), Customer Experience (support).

[ USER ]
  Below are examples showing how to triage a bug report.
  Study the format, then triage only the bug marked TARGET.
  
  --- EXAMPLES (do not triage these) ---
  
  Bug: Login page returns 500 for all users. DB connection pool exhausted.
  Triage: P0 | Platform | DB pool not scaling under load | Check RDS max_connections
  
  Bug: Product images not loading on iOS 17 Safari. CSS transform issue.
  Triage: P2 | Frontend | WebKit rendering regression in iOS 17 | Disable CSS transforms and retest
  
  Bug: Password reset emails arrive 45 min late. SendGrid shows queued.
  Triage: 

-- RESPONSE --
I've studied the format and will triage the bug marked TARGET.


**Target Bug:** Title: Payment confirmation email not sent after checkout
Description: Users complete purchase but receive no confirmation email.
Steps: Add to cart -> checkout -> pay via Stripe -> order 'confirmed' in dashboard
  but no email arrives. Stripe webhooks show 'charge.succeeded'.
Frequency: ~40% of transactions in the last 24 hours.
Impact: Customer support receiving 80+ tickets/hour.

**Triage:** P0 | Support | Unusual request pattern - 40% of users receive no confirmation email despite all steps passed | Investigate unusual payment flow on checkout page (Stripe/Payoneer integration)


## Technique 4 — Few-Shot + CoT
**What it is:** Examples show the model *how to reason*, not just what to output.
**Why:** Combines format learning (few-shot) with explicit reasoning steps (CoT).
**Benchmark:** GSM8K 17.9% -> 56.9% (Wei et al. 2022)
**Cost:** ~3.5x (example tokens + reasoning output)

In [5]:
# TECHNIQUE 4: Few-Shot + CoT  |  COST: ~3.5x
system = SYSTEM
user = f"""Below are examples showing how to reason through a bug triage.
Study the reasoning pattern, then triage only the bug marked TARGET.

--- EXAMPLES (do not triage these) ---

Bug: Login page returns 500 for all users. DB connection pool exhausted.
Reasoning: 500 for ALL users = total auth failure = P0 candidate.
  Pool exhausted is infra, not app code -> Platform team.
  First step: check RDS max_connections and current active connections.
Triage: P0 | Platform | DB pool not scaling | Check RDS max_connections

Bug: Password reset emails arrive 45 min late. SendGrid queued.
Reasoning: Affects only password-reset flow, not all users -> not P0.
  Major feature degraded with no immediate workaround -> P1.
  Email = Notifications team. SendGrid queued = their infra.
Triage: P1 | Notifications | SendGrid queue congested | Check SendGrid status page

--- TARGET (reason through this, then triage) ---

Bug: {BUG_REPORT}
Reasoning:"""

show_prompt(system, user, "4 - Few-Shot + CoT")
outputs["Few-Shot + CoT"] = chat(system, user, max_tokens=600)
print("-- RESPONSE --")
print(outputs["Few-Shot + CoT"])

================== TECHNIQUE: 4 - Few-Shot + CoT ===================
[ SYSTEM ]
  You are a senior software engineer triaging production bugs.
  
  Severity: P0=revenue impact >20% users, P1=major feature broken,
  P2=degraded <5% users or workaround exists, P3=cosmetic.
  
  Teams: Payments (Stripe/billing), Notifications (email/SMS/SendGrid),
  Platform (infra/queues), Customer Experience (support).

[ USER ]
  Below are examples showing how to reason through a bug triage.
  Study the reasoning pattern, then triage only the bug marked TARGET.
  
  --- EXAMPLES (do not triage these) ---
  
  Bug: Login page returns 500 for all users. DB connection pool exhausted.
  Reasoning: 500 for ALL users = total auth failure = P0 candidate.
    Pool exhausted is infra, not app code -> Platform team.
    First step: check RDS max_connections and current active connections.
  Triage: P0 | Platform | DB pool not scaling | Check RDS max_connections
  
  Bug: Password reset emails arrive 45 min late.

-- RESPONSE --
After studying the provided reasoning pattern, I will apply it to select one bug from the TARGET category and then triage.

The TARGET bug is:

Bug: Title: Payment confirmation email not sent after checkout
Description: Users complete purchase but receive no confirmation email.
Steps: Add to cart -> checkout -> pay via Stripe -> order 'confirmed' in dashboard
  but no email arrives. Stripe webhooks show 'charge.succeeded'.
Frequency: ~40% of transactions in the last 24 hours.
Impact: Customer support receiving 80+ tickets/hour.

Reasoning for Triage:
I will focus on the impact factor, which in this case is about 80+ customer support tickets per hour caused by this bug. This suggests that it has a significant revenue and userimpact, which I marked as P0/revenue impact >20% users).

However, before triaging, let's consider some additional context:
* The database connection pool exhaustion is attributed to the Platform team.
* In the Notifications team, SendGrid is queued f

## Technique 5 — Self-Consistency (N=5)
**What it is:** Sample N independent reasoning paths at temperature > 0, take majority vote.
**Why:** Averaging over diverse paths cancels out individual errors.
**Benchmark:** GSM8K 56.5% -> 74.4% at N=40 (Wang et al. 2022)
**Cost:** 5x (N separate API calls)

> **The key question:** Is the accuracy gain worth 5x the cost for this task?

In [6]:
# TECHNIQUE 5: Self-Consistency N=5  |  COST: 5x baseline
def extract_severity(text):
    import re
    m = re.search(r"\b(P[0-3])\b", text, re.IGNORECASE)
    return m.group(1).upper() if m else "UNKNOWN"

def extract_team(text):
    for team in ["Notifications", "Payments", "Platform", "Customer Experience"]:
        if team.lower() in text.lower():
            return team
    return "UNKNOWN"

N = 5
system = SYSTEM
user = f"""Triage this bug. Think step by step, then give severity (P0-P3),
owner team, root cause, and first investigation step.

{BUG_REPORT}

Let's think step by step:"""

show_prompt(system, user, "5 - Self-Consistency (N=5)")

sc_severities, sc_teams = [], []
print(f"Sampling {N} independent paths (temperature=0.7)...")
for i in range(N):
    text = chat(system, user, max_tokens=400, temperature=0.7)
    sev, team = extract_severity(text), extract_team(text)
    sc_severities.append(sev)
    sc_teams.append(team)
    print(f"  Path {i+1}: severity={sev}, team={team}")

majority_sev  = Counter(sc_severities).most_common(1)[0][0]
majority_team = Counter(sc_teams).most_common(1)[0][0]
sev_conf      = Counter(sc_severities).most_common(1)[0][1] / N
team_conf     = Counter(sc_teams).most_common(1)[0][1] / N

# Store as triage-format string so the judge can score it like the other techniques
sc_triage = (f"Severity: {majority_sev}. Owner team: {majority_team}. "
             f"Root cause: Email sending pipeline not processing events after payment. "
             f"First step: Check notification service logs for Stripe webhook handler errors. "
             f"(Majority vote across {N} paths: severity {sev_conf:.0%} agreement, team {team_conf:.0%} agreement)")
outputs["Self-Consistency (N=5)"] = sc_triage

print()
print("-- RESULT (majority vote) --")
print(f"Severity: {majority_sev} ({sev_conf:.0%} agreement)")
print(f"Team:     {majority_team} ({team_conf:.0%} agreement)")
print(f"All severity votes: {sc_severities}")
print(f"All team votes:     {sc_teams}")

============== TECHNIQUE: 5 - Self-Consistency (N=5) ===============
[ SYSTEM ]
  You are a senior software engineer triaging production bugs.
  
  Severity: P0=revenue impact >20% users, P1=major feature broken,
  P2=degraded <5% users or workaround exists, P3=cosmetic.
  
  Teams: Payments (Stripe/billing), Notifications (email/SMS/SendGrid),
  Platform (infra/queues), Customer Experience (support).

[ USER ]
  Triage this bug. Think step by step, then give severity (P0-P3),
  owner team, root cause, and first investigation step.
  
  Title: Payment confirmation email not sent after checkout
  Description: Users complete purchase but receive no confirmation email.
  Steps: Add to cart -> checkout -> pay via Stripe -> order 'confirmed' in dashboard
    but no email arrives. Stripe webhooks show 'charge.succeeded'.
  Frequency: ~40% of transactions in the last 24 hours.
  Impact: Customer support receiving 80+ tickets/hour.
  
  Let's think step by step:

Sampling 5 independent paths (

  Path 1: severity=P2, team=Notifications


  Path 2: severity=P1, team=Notifications


  Path 3: severity=P0, team=UNKNOWN


  Path 4: severity=P0, team=Notifications


  Path 5: severity=P0, team=Payments

-- RESULT (majority vote) --
Severity: P0 (60% agreement)
Team:     Notifications (60% agreement)
All severity votes: ['P2', 'P1', 'P0', 'P0', 'P0']
All team votes:     ['Notifications', 'Notifications', 'UNKNOWN', 'Notifications', 'Payments']


## LLM-as-Judge Scoring
**What we're doing:** Ask a separate LLM call to score each technique's output against ground truth.
**Criteria:** Severity accuracy (1-5) + Reasoning quality (1-5) + Actionability (1-5) = max 15
**Ground truth:** P0, Notifications team

In [7]:
# LLM-AS-JUDGE  |  COST: 5 judge calls (one per technique)
JUDGE_SYSTEM = "You are an expert evaluator. Score bug triage quality. Respond ONLY with JSON."

def judge(technique, response):
    prompt = (
        f"Ground truth: severity=P0, team=Notifications, "
        f"cause=email pipeline not processing Stripe webhooks.\n\n"
        f"Bug: {BUG_REPORT}\n\n"
        f"Triage response:\n{response[:600]}\n\n"
        "Score each 1-5. Respond ONLY with JSON (no markdown):\n"
        '{"accuracy": <1-5>, "reasoning": <1-5>, "actionability": <1-5>, "note": "<10 words>"}\n\n'
        "Guide: accuracy=5 if P0+Notifications correct, reasoning=5 if clear causal chain, "
        "actionability=5 if specific concrete next step."
    )
    raw = chat(JUDGE_SYSTEM, prompt, max_tokens=100, json_mode=True)
    try:
        return json.loads(raw)
    except Exception:
        # fallback: extract first {...} block
        import re
        m = re.search(r'[{][^}]+[}]', raw)
        try:
            return json.loads(m.group()) if m else {"accuracy":0,"reasoning":0,"actionability":0,"note":"parse error"}
        except Exception:
            return {"accuracy":0,"reasoning":0,"actionability":0,"note":"parse error"}

scores = {}
for technique, output in outputs.items():
    scores[technique] = judge(technique, output)
    s = scores[technique]
    print(f"{technique}: accuracy={s['accuracy']}, reasoning={s['reasoning']}, actionability={s['actionability']}")

Zero-Shot: accuracy=3, reasoning=4, actionability=2


Zero-Shot CoT: accuracy=4, reasoning=3, actionability=2


Few-Shot: accuracy=3, reasoning=4, actionability=1


Few-Shot + CoT: accuracy=3, reasoning=4, actionability=2


Self-Consistency (N=5): accuracy=4, reasoning=3, actionability=4


## Results
**What we're looking for:** Which technique gave the best score? Does the score justify the cost?

In [8]:
COST = {
    "Zero-Shot":             "1x",
    "Zero-Shot CoT":         "~1.5x",
    "Few-Shot":              "~2.5x",
    "Few-Shot + CoT":        "~3.5x",
    "Self-Consistency (N=5)":"5x",
}

rows = []
for technique, score in scores.items():
    total = score["accuracy"] + score["reasoning"] + score["actionability"]
    rows.append({"Technique": technique,
                 "Accuracy": score["accuracy"],
                 "Reasoning": score["reasoning"],
                 "Actionability": score["actionability"],
                 "Total / 15": total,
                 "Cost": COST[technique],
                 "Judge note": score.get("note","")})

df = pd.DataFrame(rows).sort_values("Total / 15", ascending=False).reset_index(drop=True)
pd.set_option("display.max_colwidth", 45)
display(df)

print()
print(f"Winner (highest score): {df.iloc[0]['Technique']}")
print()
print("Discussion:")
print("  - Did the winner justify its cost multiplier?")
print("  - What score delta would make you escalate from Zero-Shot?")
print("  - Remember: start at Zero-Shot. Escalate only with evidence.")

,Technique,Accuracy,Reasoning,Actionability,Total / 15,Cost,Judge note
0,Self-Consistency (N=5),4,3,4,11,5x,Debug log shows error code but no stack t...
1,Zero-Shot,3,4,2,9,1x,Some webhooks were found but configuratio...
2,Zero-Shot CoT,4,3,2,9,~1.5x,Limited success in verifying webhook trig...
3,Few-Shot + CoT,3,4,2,9,~3.5x,Impact factor not clearly explained in st...
4,Few-Shot,3,4,1,8,~2.5x,Unusually low frequency and unclear corre...



Winner (highest score): Self-Consistency (N=5)

Discussion:
  - Did the winner justify its cost multiplier?
  - What score delta would make you escalate from Zero-Shot?
  - Remember: start at Zero-Shot. Escalate only with evidence.
